# Model comparison

Same features, same months: train on months 0–3, measure on month 4. Month 5 (test) stays closed.

## 1. Load

The features built in `02_features.ipynb`. The test month is not even loaded as X, y: it stays closed until the end.

In [ ]:
import json, time
import numpy as np
import pandas as pd

df = pd.read_parquet("../data/processed/features.parquet")
info = json.load(open("features.json"))
FEATURES, CATEGORIES = info["features"], info["categories"]
NUMBERS = [c for c in FEATURES if c not in CATEGORIES]
for c in CATEGORIES:
    df[c] = df[c].astype("category")

train = df[df["split"] == "train"]
valid = df[df["split"] == "valid"]
X_train, y_train = train[FEATURES], train["isFraud"]
X_valid, y_valid = valid[FEATURES], valid["isFraud"]
print(f"{len(FEATURES)} features ({len(CATEGORIES)} categories, {len(NUMBERS)} numbers)")
print(f"train {len(train):,} purchases, valid {len(valid):,} purchases")

## 2. How we measure a model (on the valid month)

- **PR-AUC**: how well the model puts frauds at the top, 0 to 1. With 3.5% of frauds, a random model gets ~0.035.
- **ROC-AUC**: same idea, more common but too kind with rare classes (a random model gets 0.5).
- **top 2%**: the analysts can check 2% of the purchases. We take the 2% with the highest score and ask:
  how many of all the frauds are there, how many of those alerts are really frauds, and how many fraud dollars they cover.
- **train time**.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

ALERT_SHARE = 0.02          # the analysts can check 2% of the purchases

def evaluate(name, scores, seconds):
    """How good is a model on the valid month."""
    y = y_valid.to_numpy()
    amount = valid["TransactionAmt"].to_numpy()
    n_alerts = int(len(y) * ALERT_SHARE)
    top = np.argsort(-scores)[:n_alerts]                    # the purchases with the highest score
    return {
        "model": name,
        "PR-AUC": round(average_precision_score(y, scores), 3),
        "ROC-AUC": round(roc_auc_score(y, scores), 3),
        "frauds caught (top 2%)": f"{y[top].sum() / y.sum() * 100:.1f}%",
        "precision (top 2%)": f"{y[top].mean() * 100:.1f}%",
        "fraud $ caught (top 2%)": f"{amount[top][y[top] == 1].sum() / amount[y == 1].sum() * 100:.1f}%",
        "train time (s)": round(seconds),
    }

results, scores = [], {}

## 3. Logistic regression (the simple baseline)

It does not accept empty values or text: empty numbers are filled with the median **plus a column "was empty"**
(so the information is not lost), numbers are put on the same scale, categories become 0/1 columns.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def as_text(X):
    return X[CATEGORIES].astype(str)

prepare = ColumnTransformer([
    ("numbers", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), NUMBERS),
    ("categories", OneHotEncoder(handle_unknown="ignore", min_frequency=100), CATEGORIES),
])
logistic = make_pipeline(prepare, LogisticRegression(max_iter=500))

start = time.time()
logistic.fit(X_train.assign(**as_text(X_train)), y_train)
scores["logistic"] = logistic.predict_proba(X_valid.assign(**as_text(X_valid)))[:, 1]
results.append(evaluate("logistic regression", scores["logistic"], time.time() - start))
results[-1]

## 4. Random forest

Many independent trees, each on a random sample of purchases and columns; the score is their average.
Categories become numbers, empty values become -999 (a value apart).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def as_numbers(X):
    X = X.copy()
    for c in CATEGORIES:
        X[c] = X[c].cat.codes                               # category -> number (-1 = missing)
    return X.fillna(-999)                                   # empty -> -999 (a value "apart")

forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=50, max_features="sqrt", n_jobs=-1, random_state=0)
start = time.time()
forest.fit(as_numbers(X_train), y_train)
scores["random_forest"] = forest.predict_proba(as_numbers(X_valid))[:, 1]
results.append(evaluate("random forest", scores["random_forest"], time.time() - start))
results[-1]

## 5. XGBoost

Gradient boosting: trees in sequence, each one corrects the errors of the previous ones. Stops when the valid month stops improving (early stopping).

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    n_estimators=2000, learning_rate=0.05, max_depth=8, subsample=0.8, colsample_bytree=0.5,
    tree_method="hist", enable_categorical=True,            # histograms + native categories
    eval_metric="aucpr", early_stopping_rounds=100,         # stop when the valid month stops improving
    n_jobs=-1, random_state=0,
)
start = time.time()
xgb_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
scores["xgboost"] = xgb_model.predict_proba(X_valid)[:, 1]
results.append(evaluate("xgboost", scores["xgboost"], time.time() - start))
results[-1]

## 6. LightGBM

Gradient boosting, faster: histograms and leaf-wise growth. Accepts categories and empty values as they are.

In [ ]:
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(
    n_estimators=2000, learning_rate=0.05, num_leaves=63, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.5, metric="average_precision", n_jobs=-1, random_state=0, verbose=-1,
)
start = time.time()
lgb_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)],
              callbacks=[lgb.early_stopping(100, verbose=False)])
scores["lightgbm"] = lgb_model.predict_proba(X_valid)[:, 1]
results.append(evaluate("lightgbm", scores["lightgbm"], time.time() - start))
results[-1]

## 7. CatBoost

Gradient boosting specialised in categories (encodes them using only the previous purchases).

In [ ]:
from catboost import CatBoostClassifier

def as_catboost(X):
    X = X.copy()
    for c in CATEGORIES:
        X[c] = X[c].astype(str)                             # CatBoost wants categories as text ("nan" = missing)
    return X

cat_model = CatBoostClassifier(
    iterations=2000, learning_rate=0.05, depth=6, eval_metric="PRAUC", early_stopping_rounds=100,
    cat_features=CATEGORIES, random_seed=0, thread_count=-1, verbose=False,
)
start = time.time()
cat_model.fit(as_catboost(X_train), y_train, eval_set=(as_catboost(X_valid), y_valid))
scores["catboost"] = cat_model.predict_proba(as_catboost(X_valid))[:, 1]
results.append(evaluate("catboost", scores["catboost"], time.time() - start))
results[-1]

## 8. Comparison

One row per model. The valid scores are saved to choose the threshold in the next step.

In [ ]:
comparison = pd.DataFrame(results).set_index("model")

# the scores of every model on the valid month, for the next step (threshold in money)
saved = pd.DataFrame(scores)
saved[["TransactionID", "isFraud", "TransactionAmt"]] = valid[["TransactionID", "isFraud", "TransactionAmt"]].to_numpy()
saved.to_parquet("../data/processed/valid_scores.parquet", index=False)

comparison